# Multi-Agent Inference: Iterative Refinement System

**Purpose**: Run multi-agent iterative refinement system

**Output**: `output/multiagent_k3.csv` + `output/multiagent_k3_states.json`

## Multi-Agent Configuration:
- System: Multi-Agent Iterative Refinement
- Max Iterations (k): 3
- Agents: 6 (Generator, Evaluator, Extractor, Verifier, Instructions, Aggregator)

## Architecture:
```
Question → Query Generator → Graph Executor → Query Evaluator
                                                     ↓
                                            [Accept or Error?]
                                                     ↓
                                          [Verification Module]
                                            ↓           ↓
                                    Entity Extractor  Entity Verifier
                                            ↓           ↓
                                    Instructions Generator
                                            ↓
                                    Feedback Aggregator
                                            ↓
                                    Query Generator (refine)
                                            ↓
                                    [Loop until Accept or max k]
```

## Output Columns:
| Column | Description |
|--------|-------------|
| question_id | Question identifier |
| question | Natural language question |
| ground_truth | Ground truth Cypher query |
| complexity | Easy/Medium/Hard |
| final_query | Final generated query (after refinement) |
| first_attempt_query | Initial query (iteration 1) |
| total_iterations | Number of refinement iterations used |
| execution_success | Final query executed successfully |
| is_empty_result | Final query returned empty |
| pass_at_k | Output matches GT after k iterations |
| all_iterations | JSON: All iteration details |
| total_tokens | Total tokens across all iterations |
| elapsed_time | Total processing time |

## 1. Setup

In [ ]:
import sys
import os
import time
import csv
import json
import logging
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / '.env')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

print(f"Notebook started: {datetime.now()}")
print(f"Working directory: {Path.cwd()}")

## 2. Import Modules

In [ ]:
# Multi-agent system
from system.orchestrator import MultiAgentOrchestrator

# Agents
from agents.query_generator import QueryGenerator
from agents.query_evaluator import QueryEvaluator
from agents.entity_extractor import EntityExtractor
from agents.entity_verifier import EntityVerifier
from agents.instructions_generator import InstructionsGenerator
from agents.feedback_aggregator import FeedbackAggregator

# System components
from system.graph_executor import GraphExecutor

# Utilities
from utils.schema_loader import load_schema
from utils.prompt_loader import load_prompt_template

print("Modules imported successfully")

## 3. Configuration

In [ ]:
# Experiment configuration
CONFIG = {
    "name": "multiagent",
    "model": os.getenv("DEFAULT_MODEL", "qwen/qwen-2.5-coder-32b-instruct"),
    "schema_type": "only_paths",
    "max_iterations": int(os.getenv("MAX_ITERATIONS", 3)),
    "temperature": float(os.getenv("TEMPERATURE", 0.0)),
    "max_tokens": int(os.getenv("MAX_TOKENS", 512)),
    "rate_limit_delay": float(os.getenv("RATE_LIMIT_DELAY", 2.0)),
    "batch_size": int(os.getenv("BATCH_SIZE", 10)),
    "batch_pause": float(os.getenv("BATCH_PAUSE", 15.0)),
}

# Paths
OUTPUT_DIR = Path.cwd().parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILENAME = f"{CONFIG['name']}_k{CONFIG['max_iterations']}.csv"
OUTPUT_PATH = OUTPUT_DIR / OUTPUT_FILENAME

STATES_FILENAME = f"{CONFIG['name']}_k{CONFIG['max_iterations']}_states.json"
STATES_PATH = OUTPUT_DIR / STATES_FILENAME

print("Multi-Agent Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print(f"\nOutput:")
print(f"  CSV: {OUTPUT_PATH}")
print(f"  States: {STATES_PATH}")

## 4. Load Data

In [ ]:
# Load ground truth questions
gt_file = Path.cwd().parent / "data" / "ground_truth" / "ground_truth_52.csv"

questions = []
with open(gt_file, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        questions.append({
            "id": i + 1,
            "question": row["Pertanyaan"],
            "ground_truth": row["Cypher Query"],
            "complexity": row["Tingkat Kompleksitas"],
            "reasoning_level": row["Tingkat Penalaran"],
            "sublevel": row["Sublevel"]
        })

print(f"Loaded {len(questions)} questions")

# Show distribution
from collections import Counter
print(f"\nBy Complexity: {dict(Counter(q['complexity'] for q in questions))}")
print(f"By Reasoning: {dict(Counter(q['reasoning_level'] for q in questions))}")
print(f"By Sublevel: {dict(Counter(q['sublevel'] for q in questions))}")

## 5. Load Schema

In [ ]:
# Load schema
schema = load_schema(CONFIG["schema_type"])

print(f"Schema loaded: {CONFIG['schema_type']} ({len(schema)} chars)")
print(f"\nSchema preview:\n{schema[:300]}...")

## 6. Initialize Multi-Agent System

In [ ]:
# Initialize all agents
print("Initializing agents...")

agents = {
    'generator': QueryGenerator(
        model=CONFIG['model'],
        temperature=CONFIG['temperature'],
        max_tokens=CONFIG['max_tokens']
    ),
    'evaluator': QueryEvaluator(
        model=CONFIG['model'],
        temperature=CONFIG['temperature']
    ),
    'extractor': EntityExtractor(),
    'verifier': EntityVerifier(
        model=CONFIG['model'],
        temperature=CONFIG['temperature']
    ),
    'instructions': InstructionsGenerator(
        model=CONFIG['model'],
        temperature=CONFIG['temperature']
    ),
    'aggregator': FeedbackAggregator(
        model=CONFIG['model'],
        temperature=CONFIG['temperature']
    ),
    'executor': GraphExecutor()
}

print("  ✓ Query Generator")
print("  ✓ Query Evaluator")
print("  ✓ Entity Extractor")
print("  ✓ Entity Verifier")
print("  ✓ Instructions Generator")
print("  ✓ Feedback Aggregator")
print("  ✓ Graph Executor")

# Initialize orchestrator
orchestrator = MultiAgentOrchestrator(
    agents=agents,
    max_iterations=CONFIG['max_iterations']
)

print(f"\nMulti-Agent System initialized")
print(f"  Model: {CONFIG['model']}")
print(f"  Max iterations: {CONFIG['max_iterations']}")
print(f"  Schema: {CONFIG['schema_type']}")

## 7. Run Multi-Agent Inference

In [ ]:
from IPython.display import clear_output
from collections import defaultdict

# Storage for results
results = []
states = []

# Track refinement statistics
iterations_distribution = defaultdict(int)

def update_display(current, total, q_id, iterations, success, tokens):
    """Update progress display."""
    clear_output(wait=True)
    pct = current / total * 100
    
    # Calculate running statistics
    pass_at_k_count = sum(1 for r in results if r.get("pass_at_k"))
    exec_success_count = sum(1 for r in results if r.get("execution_success"))
    total_tokens_so_far = sum(r.get("total_tokens", 0) for r in results)
    avg_iterations = sum(r.get("total_iterations", 0) for r in results) / len(results) if results else 0
    
    print(f"Progress: {current}/{total} ({pct:.1f}%)")
    print(f"Last: Q{q_id} - iterations={iterations}, success={success}, tokens={tokens}")
    print(f"")
    print(f"Running Statistics:")
    print(f"  Pass@k: {pass_at_k_count}/{current} ({100*pass_at_k_count/current:.1f}%)")
    print(f"  Execution Success: {exec_success_count}/{current} ({100*exec_success_count/current:.1f}%)")
    print(f"  Avg Iterations: {avg_iterations:.2f}")
    print(f"  Total Tokens: {total_tokens_so_far:,}")
    
    # Show iterations distribution
    if iterations_distribution:
        print(f"\n  Iterations Distribution:")
        for k in sorted(iterations_distribution.keys()):
            count = iterations_distribution[k]
            print(f"    k={k}: {count}")

print("Starting Multi-Agent Inference...")
print("=" * 60)
start_time = datetime.now()

for i, q in enumerate(questions):
    # Batch pause
    if i > 0 and i % CONFIG["batch_size"] == 0:
        print(f"\n[Batch pause: {CONFIG['batch_pause']}s]")
        time.sleep(CONFIG["batch_pause"])
    
    q_start = time.time()
    
    try:
        # Run multi-agent system
        state = orchestrator.run(
            question=q["question"],
            schema=schema,
            question_id=q["id"]
        )
        
        elapsed = time.time() - q_start
        
        # Extract results from state
        first_attempt_query = state.iterations[0]['query'] if state.iterations else ""
        final_query = state.final_query
        total_iterations = state.total_iterations
        execution_success = state.execution_success
        is_empty_result = state.is_empty_result
        
        # Track iterations
        iterations_distribution[total_iterations] += 1
        
        # Calculate tokens
        total_tokens = sum(iter['tokens'] for iter in state.iterations)
        
        # Pass@k will be computed in metrics evaluation
        pass_at_k = False  # Placeholder
        
        # Store state
        states.append(state.to_dict())
        
        # Record result
        results.append({
            "question_id": q["id"],
            "question": q["question"],
            "ground_truth": q["ground_truth"],
            "complexity": q["complexity"],
            "reasoning_level": q["reasoning_level"],
            "sublevel": q["sublevel"],
            "final_query": final_query,
            "first_attempt_query": first_attempt_query,
            "total_iterations": total_iterations,
            "execution_success": execution_success,
            "is_empty_result": is_empty_result,
            "pass_at_k": pass_at_k,  # Will be computed later
            "all_iterations": json.dumps(state.iterations, ensure_ascii=False),
            "total_tokens": total_tokens,
            "elapsed_time": round(elapsed, 2)
        })
        
        update_display(
            i + 1, len(questions), q["id"],
            total_iterations, execution_success, total_tokens
        )
        
    except Exception as e:
        print(f"Error on Q{q['id']}: {e}")
        import traceback
        traceback.print_exc()
        
        results.append({
            "question_id": q["id"],
            "question": q["question"],
            "ground_truth": q["ground_truth"],
            "complexity": q["complexity"],
            "reasoning_level": q["reasoning_level"],
            "sublevel": q["sublevel"],
            "final_query": "",
            "first_attempt_query": "",
            "total_iterations": 0,
            "execution_success": False,
            "is_empty_result": False,
            "pass_at_k": False,
            "all_iterations": json.dumps([{"error": str(e)}]),
            "total_tokens": 0,
            "elapsed_time": 0
        })
    
    # Rate limiting
    if i < len(questions) - 1:
        time.sleep(CONFIG["rate_limit_delay"])

end_time = datetime.now()
duration = str(end_time - start_time)

print(f"\n\nMulti-agent inference completed!")
print(f"Duration: {duration}")

## 8. Save Results

In [ ]:
import pandas as pd

# Create DataFrame
df = pd.DataFrame(results)

# Save to CSV
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"Saved CSV: {OUTPUT_PATH}")
print(f"Total rows: {len(df)}")

# Save states JSON
with open(STATES_PATH, "w", encoding="utf-8") as f:
    json.dump(states, f, indent=2, ensure_ascii=False, default=str)
print(f"Saved States: {STATES_PATH}")

## 9. Quick Summary

In [ ]:
print("=" * 60)
print("MULTI-AGENT INFERENCE SUMMARY")
print("=" * 60)

total = len(results)
exec_success = sum(1 for r in results if r["execution_success"])
empty_results = sum(1 for r in results if r["is_empty_result"])
total_tokens = sum(r["total_tokens"] for r in results)
total_time = sum(r["elapsed_time"] for r in results)
avg_iterations = sum(r["total_iterations"] for r in results) / total

print(f"\nConfiguration:")
print(f"  Model: {CONFIG['model']}")
print(f"  Max k: {CONFIG['max_iterations']}")
print(f"  Schema: {CONFIG['schema_type']}")

print(f"\nExecution Statistics:")
print(f"  Total questions: {total}")
print(f"  Execution success: {exec_success}/{total} ({100*exec_success/total:.1f}%)")
print(f"  Empty results: {empty_results}/{total} ({100*empty_results/total:.1f}%)")
print(f"  Avg iterations: {avg_iterations:.2f}")

print(f"\nIterations Distribution:")
for k in sorted(iterations_distribution.keys()):
    count = iterations_distribution[k]
    pct = 100 * count / total
    print(f"  k={k}: {count} ({pct:.1f}%)")

print(f"\nCost & Performance:")
print(f"  Total tokens: {total_tokens:,}")
print(f"  Avg tokens/question: {total_tokens/total:,.0f}")
print(f"  Total time: {total_time:.1f}s")
print(f"  Avg time/question: {total_time/total:.1f}s")
print(f"  Duration: {duration}")

print(f"\nOutput:")
print(f"  CSV: {OUTPUT_PATH}")
print(f"  States: {STATES_PATH}")

print(f"\n" + "=" * 60)
print("Next: Run 03_metrics_evaluation.ipynb to compare with baseline")
print("=" * 60)

## 10. Preview Results

In [ ]:
# Preview results
display(df[["question_id", "complexity", "total_iterations",
            "execution_success", "is_empty_result"]].head(10))